# 09_trend_hypothesis — 短期トレンド仮説の検証(Google Colab / ローカル両対応)

**仮説**: ファストファッションでは、季節性(前年同週の売れ方)よりも「直近数日〜数週の流行の勢い・新しさ・同年代での流行」が購買を決める。

- 根拠1: 季節横断holdout(8 cutoff)で、0917版のYoY季節性特徴量は平均差≈0で効果なし(`00_md/pipeline_build_log.md` / Claude Project「パイプライン最新ステータス」11節)
- 根拠2: 公開されている上位解法の多くは直近1〜6週の情報だけで学習・候補生成しており、季節性を明示的に使っていない

**このノートブックでやること**(重い処理は全て `run_trend_experiment.py` をサブプロセスで呼ぶ。カーネルにメモリが溜まらない):

| 条件 | 候補プール | 特徴量 | 何を測るか |
|---|---|---|---|
| `base` | 0906版と同じ6ソース | 0906版と同じ35列 | 対照群(同じプロトコルで学習し直した0906相当) |
| `trend_feat` | 同上 | + 短期トレンド18列 | 特徴量だけの効果 |
| `trend_full` | + 年齢層別トレンド候補 | + `from_age_trend` | 候補追加の効果 |

**評価の独立性**: train2〜4で学習 / train1でearly stopping / val・holdout1〜3(と任意でseason1〜3)は報告専用。差は顧客単位のペアブートストラップで95%信頼区間を出す。

**Colabの前提**: ランタイムは **CPU + 高RAM(High-RAM, 約51GB)**。標準(12.7GB)では学習データの読み込みで足りない。
所要時間の目安: 特徴量構築 1週あたり15〜25分 × 9週(最終提出用含む)、学習+採点 1〜1.5時間。
各ステップは途中で落ちても、同じセルを再実行すれば終わったところから再開する。

## 0. 設定

In [ ]:
import glob, os, sys, shutil, subprocess

REPO_URL = "https://github.com/woojoohyeon-cpu/kaggle_hm.git"
BRANCH = "main"

# 小さい成果物(モデル・結果・提出ファイル・ログ)の保存先。前回モデルを置いたのと同じDriveフォルダ
DRIVE_DIR = "/content/drive/MyDrive/Colab Notebooks/hm"
# 特徴量parquet(1週あたり約0.7GB×9週)の置き場所。Colabのローカルディスク(ランタイムが消えると消える)
COLAB_WORK_DIR = "/content/hm_work"

# duckdbのメモリ上限。High-RAM(51GB)なら20GB。標準ランタイムでは動かさないこと
DUCKDB_MEMORY = "20GB"

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
print("IN_COLAB =", IN_COLAB)

LOG_PATH = None  # セル5で設定(Colabでは Drive 上のログにも追記する)


def sh(cmd, cwd=None):
    # 外部コマンドを実行し、出力を1行ずつこのセルに表示する(Colabでも確実に表示されるようにパイプ経由で読む)
    print("$", " ".join(cmd), flush=True)
    logf = open(LOG_PATH, "a") if LOG_PATH else None
    with subprocess.Popen(cmd, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1) as proc:
        for line in proc.stdout:
            if "it/s]" in line:   # ALSの進捗バー(大量に出る)は省略
                continue
            print(line, end="", flush=True)
            if logf:
                logf.write(line); logf.flush()
    if logf:
        logf.close()
    if proc.returncode != 0:
        hint = " (-9はOSによる強制終了=メモリ不足の可能性が高い)" if proc.returncode in (-9, 137) else ""
        raise RuntimeError(f"失敗しました returncode={proc.returncode}{hint}")

## 1. 実行環境の確認(RAMが足りるか)

In [ ]:
import psutil
total_gb = psutil.virtual_memory().total / 1e9
print(f"RAM: {total_gb:.1f} GB / CPU: {os.cpu_count()} コア")
if total_gb < 40:
    print("⚠️ RAMが40GB未満です。Colabなら『ランタイム → ランタイムのタイプを変更 → 高RAM』に切り替えてください。"
          "このまま進めると学習データ読み込み時にメモリ不足で落ちる可能性が高いです。")

## 2. Google Driveのマウントとリポジトリの取得(Colabのみ)

リポジトリは毎回 `/content/kaggle_hm` にclone(既にあれば `git pull`)する。**先にローカルでコミットしたコードをGitHubにpushしておくこと。**

In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(DRIVE_DIR, exist_ok=True)
    os.chdir("/content")
    if not os.path.exists("/content/kaggle_hm/.git"):
        sh(["git", "clone", "-b", BRANCH, REPO_URL, "/content/kaggle_hm"])
    else:
        sh(["git", "-C", "/content/kaggle_hm", "pull", "--ff-only"])
    REPO_DIR = "/content/kaggle_hm"
else:
    REPO_DIR = os.path.abspath("..")  # ローカルではZ_notebook/からこのノートブックを開いている前提

NB_DIR = os.path.join(REPO_DIR, "Z_notebook")
os.chdir(NB_DIR)
print("cwd:", os.getcwd())
sh(["git", "-C", REPO_DIR, "log", "--oneline", "-3"])

## 3. 依存ライブラリ(Colabのみ。lightgbm/pyarrow/pandasはColabに入っている)

In [ ]:
if IN_COLAB:
    sh([sys.executable, "-m", "pip", "install", "-q", "duckdb", "implicit", "kaggle"])
import duckdb, lightgbm, implicit, pandas, pyarrow
print("duckdb", duckdb.__version__, "/ lightgbm", lightgbm.__version__, "/ implicit", implicit.__version__,
      "/ pandas", pandas.__version__, "/ pyarrow", pyarrow.__version__)

## 4. Kaggle認証とデータのダウンロード(未取得のファイルだけ)

トークンはノートブックに直接書かず、Colab左側の🔑(シークレット)に `KAGGLE_API_TOKEN` という名前で登録しておく
(値は kaggle.com/settings の「API Tokens」で発行した `KGAT_...` の文字列。ノートブックへのアクセスを許可するのを忘れずに)。
ローカルでは `~/.kaggle/access_token` が既にある前提。

In [ ]:
token_path = os.path.expanduser("~/.kaggle/access_token")
if IN_COLAB and not os.path.exists(token_path):
    from google.colab import userdata
    token = userdata.get("KAGGLE_API_TOKEN")
    os.makedirs(os.path.dirname(token_path), exist_ok=True)
    with open(token_path, "w") as f:
        f.write(token.strip())
    os.chmod(token_path, 0o600)
print("Kaggleトークン:", "OK" if os.path.exists(token_path) else "見つかりません")

INPUT_DIR = os.path.join(REPO_DIR, "01_input")
os.makedirs(INPUT_DIR, exist_ok=True)
COMP = "h-and-m-personalized-fashion-recommendations"
for fname in ["articles.csv", "customers.csv", "transactions_train.csv", "sample_submission.csv"]:
    if os.path.exists(os.path.join(INPUT_DIR, fname)):
        print(f"[skip] {fname}")
        continue
    sh(["kaggle", "competitions", "download", "-c", COMP, "-f", fname, "-p", INPUT_DIR])
    z = os.path.join(INPUT_DIR, fname + ".zip")
    if os.path.exists(z):
        sh(["unzip", "-o", "-q", z, "-d", INPUT_DIR])
        os.remove(z)
    print(f"[ok] {fname}")
sh(["ls", "-la", INPUT_DIR])

## 5. 環境変数(duckdbのメモリ・作業ディレクトリ・同期先)と前処理(trans_lite / テキスト類似度)

In [ ]:
os.environ["HM_DUCKDB_MEMORY"] = DUCKDB_MEMORY
os.environ["HM_DUCKDB_THREADS"] = str(os.cpu_count())
if IN_COLAB:
    os.environ["HM_DUCKDB_TMP"] = "/content/duckdb_tmp"          # スピル先はローカルディスク
    os.environ["HM_WORK_DIR"] = COLAB_WORK_DIR
    os.environ["HM_SYNC_DIR"] = os.path.join(DRIVE_DIR, "trend_experiment")
else:
    os.environ.pop("HM_SYNC_DIR", None)                          # ローカルは 02_output/experiments/trend
    os.environ.pop("HM_WORK_DIR", None)
work_dir = os.environ.get("HM_WORK_DIR", os.path.join(REPO_DIR, "02_output", "experiments", "trend"))
print({k: v for k, v in os.environ.items() if k.startswith("HM_")}, "work_dir =", work_dir)
if IN_COLAB:
    os.makedirs(os.environ["HM_SYNC_DIR"], exist_ok=True)
    LOG_PATH = os.path.join(os.environ["HM_SYNC_DIR"], "run.log")   # ブラウザを閉じてもDriveで進捗を確認できる
    print("ログ:", LOG_PATH)

sys.path.insert(0, NB_DIR)
from common import ensure_trans_lite, FEATURES_DIR
ensure_trans_lite()
sim_path = os.path.join(FEATURES_DIR, "content_similarity.parquet")
if not os.path.exists(sim_path):
    sh([sys.executable, "build_content_similarity.py", "--out", sim_path])
else:
    print("[skip] content_similarity.parquet")

## 6. STEP 1: 特徴量テーブルの構築(学習・評価に必要な5週)

1週ずつ別プロセスで作る(1週の処理が終わるたびにメモリがOSに返る)。途中で止まっても、再実行すれば作成済みの週は飛ばす。
メモリ不足でOSに強制終了された場合は、黙って消えずに `returncode=-9 ... メモリ不足の可能性が高い` と表示される。

In [ ]:
def run(*args):
    sh([sys.executable, "-u", "run_trend_experiment.py", *args])

run("build", "--only", "val,train1,train2,train3,train4")

## 7. STEP 2: 3条件の学習と評価(val週)

学習済みの条件は飛ばして評価だけ行う。最後にレポート(週別MAP@12・ブートストラップ信頼区間・特徴量重要度)が出る。

In [ ]:
run("train")

## 8. STEP 3: holdout 3週を追加して再評価(週ごとのばらつきを見る)

学習はやり直さず、追加した週の採点だけ行う。1週だけの結果で判断しないためのステップ(これまでoptuna・候補G・季節性特徴量で
「1週では良く見えたがLBで悪化」を3回経験している)。

In [ ]:
run("build", "--only", "holdout1,holdout2,holdout3")
run("train")

## 8b. (任意)季節の異なる3週でも採点する(2019-09-21 秋 / 2019-12-21 冬 / 2020-06-20 夏)

前回の季節横断検証と同じ日付。短期トレンド特徴量の効果が「今の季節だけ」のものでないかを確認する(0917版の季節性特徴量は
7〜8月でだけ効いて他の季節では効かず、Private LBで悪化した)。3週分の構築で+1時間ほどかかる。

In [ ]:
RUN_SEASON_WEEKS = False
if RUN_SEASON_WEEKS:
    run("build", "--only", "season1,season2,season3")
    run("train")

## 9. (任意)現行ベスト0906版モデルも同じ週で採点して並べる

前回Driveに置いた `model_20260906_0924_ranker.pkl` と `train_categories_0906.pkl` を使う。
0906版はval週でearly stoppingしているため、val週の数字は楽観的(holdout週の比較の方が公平)。

In [ ]:
ref_model = os.path.join(DRIVE_DIR, "model_20260906_0924_ranker.pkl") if IN_COLAB else \
    os.path.join(REPO_DIR, "02_output", "models", "model_20260906_0924_ranker.pkl")
ref_cats = os.path.join(os.path.dirname(ref_model), "train_categories_0906.pkl")
if os.path.exists(ref_model) and os.path.exists(ref_cats):
    run("score-ref", "--model", ref_model, "--categories", ref_cats, "--label", "ref_0906")
else:
    print("0906版のファイルが見つからないのでスキップ:", ref_model, ref_cats)

## 10. レポートの表示

In [ ]:
from IPython.display import Markdown, display
with open(os.path.join(work_dir, "results", "report.md")) as f:
    display(Markdown(f.read()))

## 11. STEP 4: 提出ファイルの作成

レポートを見て採用する条件を決めてから実行する。判断の目安:

- `trend_full` / `trend_feat` の **pooled 95%CI下限 > 0** かつ **4分の3以上の週で改善** → 採用して提出
- CIが0をまたぐ → 差はノイズと区別できない。提出して確かめるのは良いが、LBの±0.0002程度の差で結論を出さない

`refit` は評価で決まった木の本数を固定し、直近4週(val, train1〜3)で学習し直す。
`--cold-start age_trend` は、モデルの対象外の顧客(直近56日に購入なし。全顧客の半分以上)に「同じ年齢層で直近7日に売れている上位12件」を出す
(`global` にすると従来どおり全体の上位12件)。レポートの `+agecold` 列がこの設定での評価値なので、先にそちらで効果を確認できる。

In [ ]:
CHOSEN = "trend_full"        # レポートを見て決める: base / trend_feat / trend_full
COLD_START = "age_trend"     # age_trend / global

run("refit", "--variant", CHOSEN)
run("predict", "--variant", CHOSEN, "--cold-start", COLD_START)

## 12. (任意)Kaggleへ提出してスコアを確認

In [ ]:
SUBMIT = False   # Trueにすると最新の提出ファイルを提出する

subs = sorted(glob.glob(os.path.join(work_dir, "submission", "submission_*.csv")))
print("最新:", subs[-1] if subs else None)
if SUBMIT and subs:
    sh(["kaggle", "competitions", "submit", "-c", COMP, "-f", subs[-1],
        "-m", f"trend {CHOSEN} cold_start={COLD_START}"])
if shutil.which("kaggle"):
    sh(["kaggle", "competitions", "submissions", "-c", COMP])

## 13. (任意)特徴量parquetをDriveにバックアップ/復元

Colabのランタイムが切れる(リセットされる)と `/content/hm_work/features` は消え、STEP 1からやり直しになる。
Driveに約7GBの空きがあれば、STEP 1・3の後にバックアップしておくと、次回はここから復元して続きから始められる。

In [ ]:
BACKUP = os.path.join(DRIVE_DIR, "trend_experiment", "features_backup")
ACTION = None   # "backup" / "restore" / None

src_dir = os.path.join(COLAB_WORK_DIR, "features")
if ACTION == "backup":
    os.makedirs(BACKUP, exist_ok=True)
    for f in glob.glob(os.path.join(src_dir, "*.parquet")):
        dst = os.path.join(BACKUP, os.path.basename(f))
        if not os.path.exists(dst):
            shutil.copy2(f, dst); print("backup:", os.path.basename(f))
elif ACTION == "restore":
    os.makedirs(src_dir, exist_ok=True)
    for f in glob.glob(os.path.join(BACKUP, "*.parquet")):
        dst = os.path.join(src_dir, os.path.basename(f))
        if not os.path.exists(dst):
            shutil.copy2(f, dst); print("restore:", os.path.basename(f))
    manifest_src = os.path.join(DRIVE_DIR, "trend_experiment", "manifest.json")
    if os.path.exists(manifest_src):
        shutil.copy2(manifest_src, os.path.join(COLAB_WORK_DIR, "manifest.json"))
    for sub in ["models", "results"]:
        sub_src = os.path.join(DRIVE_DIR, "trend_experiment", sub)
        if os.path.exists(sub_src):
            shutil.copytree(sub_src, os.path.join(COLAB_WORK_DIR, sub), dirs_exist_ok=True)